# Práctica guiada — De formulario a experiencia de inferencia

En S5 ya construisteis un formulario Streamlit que recoge los 11 atributos y llama a un gateway basado en el contrato de S4. En S6 **no se rediseña el contrato ni se vuelve a implementar el formulario**. El objetivo es convertir esa pantalla funcional en una interfaz operable: mantiene el resultado al haber un rerun, hace visible lo que está ocurriendo y enseña la información justa para interpretar e investigar una predicción.

Trabajad en parejas. Completad los `TODO`, ejecutad las comprobaciones y conservad el notebook como evidencia para el taller. La implementación queda para la Clase 2.

## Entrega de la práctica

Al terminar debéis poder enseñar cuatro decisiones concretas:

1. qué permanece en la sesión, qué se cachea y qué no se retiene;
2. la máquina `idle → loading → success/error`;
3. un plano de pantalla con componentes Streamlit concretos;
4. mensajes seguros para confianza baja, lentitud y error.

No se entrega código de inferencia nuevo. S4 sigue siendo la única frontera de validación y predicción.

## 1. Comprobar el punto de partida de S5

Ejecutad la celda. Debe demostrar que partís del formulario y gateway de S5, no de una segunda implementación. Después anotad una limitación que observaríais al cambiar un widget o al enviar una segunda predicción.

In [1]:
from pathlib import Path

repo_root = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / 'semana5/modules/05-streamlit-basic-model-ui').is_dir()
)
s5_app = repo_root / 'semana5/modules/05-streamlit-basic-model-ui/solutions/02-first-streamlit/app.py'
source = s5_app.read_text(encoding='utf-8')

checks = {
    'reutiliza st.form': 'st.form(' in source,
    'recoge valores antes de inferir': 'collect_values' in source,
    'aún no usa session_state': 'st.session_state' not in source,
}
print('app de S5:', s5_app)
for label, passed in checks.items():
    print(f"{'✓' if passed else '✗'} {label}")
assert all(checks.values())

# TODO: escribid aquí una limitación observable de S5.
limitacion_s5 = (
    'En S5 el resultado solo se dibuja en la ejecución en la que submitted es True: '
    'cualquier otra interacción repite el script, submitted pasa a False y el resultado '
    'se sustituye por el mensaje inicial. Además run_configured_app llama a '
    'gateway_factory() en cada ejecución, así que el bundle se recarga en cada rerun.'
)
print('Limitación observada:', limitacion_s5)

app de S5: /sessions/rcw-01qvp8cpumxwfrg54kc92bda/mnt/semana5/modules/05-streamlit-basic-model-ui/solutions/02-first-streamlit/app.py
✓ reutiliza st.form
✓ recoge valores antes de inferir
✓ aún no usa session_state
Limitación observada: En S5 el resultado solo se dibuja en la ejecución en la que submitted es True: cualquier otra interacción repite el script, submitted pasa a False y el resultado se sustituye por el mensaje inicial. Además run_configured_app llama a gateway_factory() en cada ejecución, así que el bundle se recarga en cada rerun.


### Inspección de S5 (tarea 01)

Lectura de `semana5/.../solutions/02-first-streamlit/app.py`:

| Qué buscaba | Dónde está | Qué implica para un rerun |
| --- | --- | --- |
| Formulario | `collect_values` (`st.form("wine_quality_form")`, devuelve `(submitted, values)`) | `submitted` es `True` solo en la ejecución del clic. |
| Gateway | `build_gateway` / `run_configured_app` (`gateway_factory()`) | Se vuelve a construir en cada ejecución. |
| Llamada a `predict` | `run_app`: `gateway.predict(values)` | Se hace solo si `submitted`; el resultado vive en una variable local. |
| Dónde se pierde el resultado | `run_app`: si `not submitted` → `st.info(...)` y `return` | Cualquier rerun posterior borra el resultado visible. |

El diseño de S6 parte de esta limitación y **no** copia ni rediseña el formulario: se conserva `collect_values`.

## 2. Decidir dónde vive cada cosa

Un rerun vuelve a ejecutar el script. Una variable local desaparece; `st.session_state` conserva información de **esa sesión**; `st.cache_resource` reutiliza recursos como un gateway o modelo. Un recurso cacheado global puede compartirse entre usuarios, así que debe ser seguro para concurrencia.

Completad la tabla y justificad cada decisión. No guardéis secretos ni las 11 features en la telemetría. Los valores del último envío solo se conservan para permitir un reintento y no se muestran ni se registran.

| Clave o recurso | Dónde vive | ¿Sobrevive a un rerun? | ¿Se limpia al pulsar «Limpiar»? | Motivo |
| --- | --- | --- | --- | --- |
| `last_state` | `st.session_state` (objeto `UiState`: fase, `request_id`, vista o error) | Sí, dentro de la sesión | Sí: vuelve a `idle` | Mantiene visible el último resultado tras tocar otro widget. Es de una persona; no se comparte. |
| `last_values` | `st.session_state` (diccionario con las 11 features del último envío) | Sí, dentro de la sesión | Sí: pasa a `{}` | Solo sirve para «Reintentar». No se muestra ni se registra, y se borra al limpiar para no retener datos de más. |
| `telemetry` | `st.session_state` (objeto `Telemetry` con agregados) | Sí, dentro de la sesión | No | Solo cuenta peticiones, errores por código, latencia media y p95 y versiones. No guarda features ni resultados, por eso puede sobrevivir a «Limpiar». |
| gateway / bundle | `st.cache_resource` (proceso del servidor, compartido) | Sí, también entre sesiones | No | Cargar el bundle es caro y el objeto solo se lee, así que es seguro compartirlo. Argumento de la caché: la ruta configurada (`MODEL_UI_BUNDLE`). |
| respuesta de una petición | Variable local del script; resumida en `last_state` | La variable no: se pierde al rerun | La variable desaparece sola; el resumen en `last_state` se limpia | Es dato de una persona: nunca va a `cache_resource`. Solo se conserva en la sesión de esa persona. |

Referencias: [`st.session_state`](https://docs.streamlit.io/develop/api-reference/caching-and-state/st.session_state) y [`st.cache_resource`](https://docs.streamlit.io/develop/api-reference/caching-and-state/st.cache_resource).

### Contrato de sesión y decisión de caché

- **Contrato de sesión:** tres claves (`last_state`, `last_values`, `telemetry`), creadas con `setdefault` para que un rerun no las reinicie. `session_state` guarda estado de la sesión; no es un almacén de datos.
- **Qué se cachea:** la función que construye el gateway (`build_gateway(bundle_value)`), envuelta con `st.cache_resource`. El único argumento es la ruta del bundle (o `None` para el gateway demo).
- **Qué se excluye:** los valores del formulario y la predicción de una persona. No pueden ser argumento ni resultado de una caché compartida.
- **Qué no se registra en telemetría:** las 11 features, el resultado de cada petición y secretos o rutas internas.
- **Límite:** si la carga del bundle falla, el gateway no disponible también queda en caché; arreglar la ruta exige limpiar la caché o reiniciar la app.

## 3. Diseñar la pantalla, no solo el estado

La interfaz debe tener cuatro zonas, en este orden:

1. **acción:** el formulario heredado de S5;
2. **operación:** qué está ocurriendo mientras se ejecuta;
3. **resultado:** categoría, confianza y latencia;
4. **trazabilidad:** versiones y `request_id`, visibles solo si se necesitan.

Elegid componentes que tengan una función. `st.columns` organiza, `st.status` comunica una operación, `st.metric` destaca un dato y `st.expander` aplaza el detalle técnico. No uséis una barra de progreso para la confianza: representa avance de una tarea, no certeza del modelo.

In [2]:
# Sustituid cada TODO por el componente y la justificación que usaríais.
# Las claves describen zonas de la interfaz; no son nuevos contratos de ML.

ui_plan = {
    'formulario': {
        'component': 'st.form',
        'reason': 'Agrupa los 11 campos de S5 (collect_values) y solo envía al pulsar el botón, sin una inferencia por widget.',
    },
    'estado_operacion': {
        'component': 'st.status',
        'reason': 'Comunica «loading» en un hueco reservado (st.empty) sin quitar el formulario; sirve para una operación, no para un valor.',
    },
    'resumen_resultado': {
        'component': 'st.metric',
        'reason': 'Tres métricas en st.columns(3): categoría, confianza y latencia se leen de un vistazo y no se mezclan.',
    },
    'mensaje_interpretacion': {
        'component': 'st.warning',
        'reason': 'Avisa si la confianza es baja o la respuesta lenta; con confianza media o alta se usa st.info. La confianza va en métrica y texto, no en barra de progreso, porque no es avance de una tarea.',
    },
    'trazabilidad': {
        'component': 'st.expander',
        'reason': 'Aplaza versiones del modelo y del preprocesado y el request_id: están disponibles sin invadir el resultado.',
    },
    'separacion_telemetria': {
        'component': 'st.divider',
        'reason': 'Separa la telemetría agregada de la sesión (st.json de agregados) del resultado de la petición actual.',
    },
}

allowed_components = {
    'st.form', 'st.status', 'st.empty', 'st.columns', 'st.container',
    'st.metric', 'st.success', 'st.info', 'st.warning', 'st.error',
    'st.expander', 'st.divider', 'st.caption',
}
required_zones = {
    'formulario', 'estado_operacion', 'resumen_resultado',
    'mensaje_interpretacion', 'trazabilidad', 'separacion_telemetria',
}

assert set(ui_plan) == required_zones
for zone, decision in ui_plan.items():
    assert set(decision) == {'component', 'reason'}, zone

pending = [zone for zone, decision in ui_plan.items() if 'TODO' in decision.values()]
invalid = [
    f"{zone}: {decision['component']}"
    for zone, decision in ui_plan.items()
    if 'TODO' not in decision['component'] and decision['component'] not in allowed_components
]
print('Zonas pendientes:', pending or 'ninguna')
print('Componentes no permitidos:', invalid or 'ninguno')
print('Comprobación final: pending == [] and invalid == []')

Zonas pendientes: ninguna
Componentes no permitidos: ninguno
Comprobación final: pending == [] and invalid == []


### Plano mínimo esperado

```text
Título y contexto breve

[ formulario de S5 en dos columnas ]

[ estado de la operación ]

[ Categoría ]  [ Confianza ]  [ Latencia ]
mensaje interpretativo de confianza y latencia
▶ Trazabilidad: versiones y request_id

────────────────────────────────────
Telemetría agregada de la sesión
```

Podéis entregar este plano en el notebook o redibujarlo en draw.io. Justificad cualquier cambio respecto a esta estructura.

## 4. Máquina de estados y recuperación

Completad la tabla. Una respuesta válida con latencia superior a 300 ms sigue en `success`: añade un aviso técnico, no se convierte en error. La confianza es una señal para revisar, no un diagnóstico ni una garantía.

| Estado | Evento de entrada | Zona de pantalla que cambia | Qué se muestra | Acción permitida | Evidencia |
| --- | --- | --- | --- | --- | --- |
| `idle` | Inicio / limpiar | Operación y resultado (hueco `st.empty`) | `st.info`: «Completa el formulario y pulsa «Ejecutar inferencia».» | Rellenar y enviar el formulario | Test de sesión (limpiar vuelve a `idle`); recorridos 1 y 3 de `AppTest`. |
| `loading` | Enviar formulario (o «Reintentar») | Operación | `st.status`: «Validando la muestra y ejecutando el modelo…»; el formulario sigue visible | Esperar | Test del controlador (emite `loading` antes del final); captura de loading. |
| `success` | Payload válido | Resultado y trazabilidad | Tres métricas, aviso de confianza, aviso técnico si pasa de 300 ms, expander «Trazabilidad» | Reenviar o «Limpiar último resultado» | Tests del controlador y de presentación; recorridos 2, 4 y 8. |
| `error` | Excepción controlada | Operación y resultado | Título, mensaje, recuperación y `request_id` | Corregir y reenviar, «Reintentar» o «Limpiar» | Test de traducción de errores; recorridos 5 y 6. |

Incluid una recuperación para entrada inválida, bundle no disponible y timeout. Indicad también qué no debe ver la persona: traceback, rutas internas, secretos o el payload completo.

### Recuperación por tipo de fallo

| Fallo | Código | Qué ve la persona | Recuperación ofrecida |
| --- | --- | --- | --- |
| Entrada inválida | `invalid_input` | «Revisa los datos». La muestra no cumple el contrato de entrada. | Corregir los valores y volver a enviar. |
| Bundle ausente | `artifact_unavailable` | «Modelo no disponible». | Comprobar `MODEL_UI_BUNDLE` y pulsar «Reintentar». |
| Timeout o fallo transitorio | `timeout` / `prediction_error` | «La respuesta tarda demasiado» o «No se pudo completar la inferencia». | «Reintentar» (reutiliza solo el último envío); si persiste, comunicar el `request_id`. |
| Limpiar | — | Vuelve a `idle`. | Borra `last_state` y `last_values`; **no** borra la telemetría. |

**Qué no debe ver la persona:** traceback, rutas internas, secretos ni el payload completo. El mensaje nunca incluye el texto de la excepción; solo el copy traducido y un `request_id`.

## 5. Escribir el copy de una interfaz responsable

Cada mensaje debe decir qué ha ocurrido, qué significa y qué puede hacer la persona. No debéis prometer que una predicción es correcta. Completad textos de una frase, sin terminología interna.

In [3]:
ux_copy = {
    'low_confidence': 'Confianza baja: el resultado es orientativo y conviene revisarlo; la confianza no es una garantía.',
    'slow_success': 'La respuesta llegó, pero tardó más de lo esperado; el resultado sigue siendo válido.',
    'invalid_input': 'Algún valor del formulario no es válido: corrígelo y vuelve a enviar la petición.',
    'artifact_unavailable': 'No podemos usar el modelo en este momento; comprueba la configuración y pulsa «Reintentar».',
}

assert set(ux_copy) == {
    'low_confidence', 'slow_success', 'invalid_input', 'artifact_unavailable'
}
for scenario, message in ux_copy.items():
    assert isinstance(message, str) and message.strip(), scenario

pending_copy = [scenario for scenario, message in ux_copy.items() if 'TODO' in message]
print('Mensajes pendientes:', pending_copy or 'ninguno')
print('Revisión humana: ¿el texto evita certeza, traceback y culpabilizar a la persona?')

Mensajes pendientes: ninguno
Revisión humana: ¿el texto evita certeza, traceback y culpabilizar a la persona?


## 6. Preparar el taller

Entregad el notebook con los TODO completados. Antes de cerrar, otra pareja debe poder responder estas preguntas leyendo vuestro trabajo:

- ¿Qué se mantiene al pulsar otro widget y qué se elimina con «Limpiar último resultado»?
- ¿Qué función cargará el gateway una sola vez y cuál será su límite?
- ¿Qué componentes dibujará `render_state()` para `loading`, `success` y `error`?
- ¿Qué ve una persona con confianza baja o una respuesta lenta?
- ¿Dónde aparecen el `request_id` y las versiones sin invadir el resultado?

En el taller se implementarán estas decisiones en `session.py`, `policies.py`, `presentation.py`, `controller.py` y `app.py`.

## 7. Plan de implementación para la práctica 6.2

| Requisito del diseño | Función (archivo) | Prueba en el starter 6.2 |
| --- | --- | --- |
| Sesión idempotente con `setdefault` | `initialize_session_state` (`session.py`) | `test_session_state_initialization_is_idempotent` |
| Limpiar vuelve a `idle` sin borrar telemetría | `clear_last_state` (`session.py`) | `test_clear_last_state_keeps_telemetry` |
| Umbrales de confianza explícitos (0,60 y 0,85) | `classify_confidence` (`policies.py`) | `test_confidence_policy_has_explicit_boundary_values` |
| Latencia lenta sigue siendo `success` con aviso | `classify_latency`, `build_prediction_view` (`policies.py`, `presentation.py`) | `test_latency_and_confidence_are_explained_to_the_user` |
| Errores sin detalle técnico, con recuperación y `request_id` | `to_user_facing_error` (`presentation.py`) | `test_error_translation_does_not_expose_technical_detail` |
| `loading` → `success`, con versiones y solo agregados | `PredictionController.submit` (`controller.py`) | `test_controller_emits_loading_then_success_and_records_versions` |
| `loading` → `error` sin payload en telemetría | `PredictionController.submit` | `test_controller_emits_loading_then_error_without_payload_telemetry` |
| Gateway cacheado una sola vez | `get_cached_gateway` (`app.py`) | No hay test original: se comprueba con `AppTest` (recorrido 7) |
| Dibujo por fase | `render_state` (`app.py`) | No hay test original: `AppTest` (recorridos 1 a 6) y capturas |

Orden recomendado: sesión → políticas → presentación → controlador → `app.py`, ejecutando `pytest` tras cada paso.

## Debrief

- ¿Qué limita a S5 aunque el formulario ya funcione?
- ¿Por qué el gateway es un recurso cacheable y el resultado de una petición pertenece a una sesión?
- ¿Qué componente hace visible cada transición sin añadir una segunda lógica de inferencia?
- ¿Por qué categoría, confianza y latencia no tienen el mismo significado?
- ¿Qué parte de este diseño se conserva al sustituir el gateway local por HTTP en S7?

### Respuestas

1. **Qué limita a S5:** el resultado solo existe en la ejecución del clic; cualquier rerun lo borra, y el bundle se recarga cada vez.
2. **Gateway vs. resultado:** el gateway es de solo lectura y caro de crear, así que puede compartirse (`cache_resource`); el resultado es dato de una persona y debe quedarse en su sesión.
3. **Qué hace visible cada transición:** `st.status` para `loading`, `st.metric` y avisos para `success`, `st.error` con recuperación para `error`, siempre en el mismo hueco `st.empty`.
4. **Por qué no significan lo mismo:** la categoría es la salida del modelo; la confianza es una señal para revisar, no una garantía; la latencia es del servicio y no dice nada sobre la calidad de la predicción.
5. **Qué se conserva con HTTP (S7):** estado de sesión, máquina de estados, políticas y copy. Solo cambia el gateway, que pasaría a llamar por HTTP, con timeouts y errores traducidos a los mismos códigos.